# Week 12 Assignment #7: NY Times API, PIT AI Community Access
**Student:** Daniel Foulen  
**Class:** IS 362  
**Date:** 4/17/2026

## Background
This notebook queries the New York Times Article Search API to retrieve and structure recent coverage of AI equity and community technology access. The NYT API provides programmatic access to article metadata dating back to 1851, returned as JSON.

I work at El Puente through their Digital Justice framework, so I will be taking that focus for this assignment for Public Interest Technology in regards to AI and community access.

## Applications
Structured news data like this can be used to track how mainstream media covers topics relevant to underserved communities. This includes the frequency, framing, and sourcing of coverage on AI access gaps, digital equity, and community-based technology initiatives. For a community tech lab, this kind of data pipeline supports narrative research and public advocacy work.

## Step 1: Import Libraries

We use three standard libraries only. 
No third-party NYT wrappers.

- `requests` handles the HTTP call to the NYT API.
- `pandas` structures the returned JSON into a tabular DataFrame.
- `getpass` prompts for the API key interactively without echoing it to the screen or embedding it in the notebook.

In [22]:
import requests
import pandas as pd
from getpass import getpass

## Step 2: Accept the API Key Securely

API keys should never be hardcoded in notebooks. I do not like exposing my API keys.

`getpass()` reads the key from a masked prompt at runtime and stores it in memory only for the duration of the session. The key is never written to disk.

In [23]:
api_key = getpass("Enter your NYT API key: ")

## Step 3: Query the NYT Article Search API

The Article Search API endpoint accepts query parameters via the URL. We pass:

- `q`: the search term (`"AI equity"`)
- `api-key`: our key for authentication

The API returns a JSON envelope where the actual article list lives at `response["response"]["docs"]`. We check the HTTP status code before proceeding so any authentication or quota error surfaces immediately rather than silently producing empty data.

In [24]:
BASE_URL = "https://api.nytimes.com/svc/search/v2/articlesearch.json"

params = {
    "q": "AI equity",
    "api-key": api_key,
}

response = requests.get(BASE_URL, params=params)

# Raise immediately on 4xx/5xx so we don't silently process empty data
response.raise_for_status()

data = response.json()
print(f"Status: {data['status']}: {len(data['response']['docs'])} articles returned")

Status: OK: 10 articles returned


## Step 4: Extract the Docs List and Load into a DataFrame

The JSON response wraps article records inside two levels of nesting: `response -> response -> docs`. Each doc is a dictionary with fields like `headline`, `byline`, `pub_date`, `section_name`, `snippet`, and `web_url`.

Loading the list directly into `pd.DataFrame` gives us one row per article. The `headline` and `byline` columns arrive as nested dicts, which we flatten in the next step.

In [25]:
docs = data["response"]["docs"]

df = pd.DataFrame(docs)

print(f"Shape: {df.shape}")
df.columns.tolist()

Shape: (10, 19)


['abstract',
 'byline',
 'document_type',
 'headline',
 '_id',
 'keywords',
 'multimedia',
 'news_desk',
 'print_page',
 'print_section',
 'pub_date',
 'section_name',
 'snippet',
 'source',
 'subsection_name',
 'type_of_material',
 'uri',
 'web_url',
 'word_count']

## Step 5: Flatten Nested Columns

Two columns arrive as dicts rather than scalar values:

- `headline`: a dict with keys like `"main"`, `"kicker"`, `"print_headline"`. We want the `"main"` key only.
- `byline`: a dict with a `"original"` key containing the formatted author string (e.g. `"By JANE DOE"`).

We use `.apply(lambda x: x.get(...))` instead of direct key access because individual articles occasionally omit a key, and `.get()` returns `None` rather than raising a `KeyError`.

In [26]:
# Extract the plain headline string from the nested headline dict
df["headline"] = df["headline"].apply(lambda x: x.get("main") if isinstance(x, dict) else None)

# Extract the formatted byline string; byline can be None for wire stories
df["byline"] = df["byline"].apply(lambda x: x.get("original") if isinstance(x, dict) else None)

## Step 6: Drop Columns That Add Noise

`multimedia` contains a list of image objects. Useful for rendering, but not for text analysis. `keywords` contains a list of subject/person/location tag dicts that require further unnesting to be useful and are out of scope here.

Dropping both now keeps the DataFrame readable. We use `errors="ignore"` in case either column is absent for a given API response shape.

In [27]:
df.drop(columns=["multimedia", "keywords"], inplace=True, errors="ignore")

## Step 7: Display the Cleaned DataFrame

The final DataFrame has one row per article with scalar values in every column, ready for downstream analysis: frequency counts by section, date-range filtering, sentiment scoring on the snippet, or CSV export for a spreadsheet workflow.

In [28]:
# Show all columns; widen output so headlines aren't truncated
pd.set_option("display.max_colwidth", 120)

df

,abstract,byline,document_type,headline,_id,news_desk,print_page,print_section,pub_date,section_name,snippet,source,subsection_name,type_of_material,uri,web_url,word_count
0,"OpenAI, Anthropic, Waymo and other artificial intelligence companies hauled in $297 billion in funding in the first ...",By Erin Griffith,article,"A.I. Companies Shatter Fund-Raising Records, as Boom Accelerates",nyt://article/6e56a482-0725-55f6-b138-cd33206d4285,Business,6,B,2026-04-01T18:13:37Z,Technology,"OpenAI, Anthropic, Waymo and other artificial intelligence companies hauled in $297 billion in funding in the first ...",The New York Times,,News,nyt://article/6e56a482-0725-55f6-b138-cd33206d4285,https://www.nytimes.com/2026/04/01/technology/ai-companies-fund-raising-records.html,461
1,The valuations of some artificial intelligence companies are approaching those of the dot-com boom. But investors wo...,By Joe Rennison,article,Wall Street Is Shaking Off Fears of an A.I. Bubble. For Now.,nyt://article/3516911b-597a-5e35-8b1f-5c92da525a3b,Business,1,A,2025-12-09T16:36:14Z,Business,The valuations of some artificial intelligence companies are approaching those of the dot-com boom. But investors wo...,The New York Times,,News,nyt://article/3516911b-597a-5e35-8b1f-5c92da525a3b,https://www.nytimes.com/2025/12/09/business/wall-street-valuation-ai-bubble.html,1458
2,New court cases seek to define content created by artificial intelligence as defamatory — a novel concept that has c...,By Ken Bensinger,article,Who Pays When A.I. Is Wrong?,nyt://article/19d43840-0e85-5761-8a4b-ef256cd67929,Business,1,A,2025-11-12T10:01:56Z,Business,New court cases seek to define content created by artificial intelligence as defamatory — a novel concept that has c...,The New York Times,Media,News,nyt://article/19d43840-0e85-5761-8a4b-ef256cd67929,https://www.nytimes.com/2025/11/12/business/media/ai-defamation-libel-slander.html,1614
3,"With executive orders and an “A.I. Action Plan” to promote American dominance of the technology, President Trump dec...",By David McCabe and Cecilia Kang,article,Trump Plans to Give A.I. Developers a Free Hand,nyt://article/cf5ad199-d749-52e8-ba54-3e2009838201,Business,1,A,2025-07-23T14:59:39Z,Technology,"With executive orders and an “A.I. Action Plan” to promote American dominance of the technology, President Trump dec...",The New York Times,,News,nyt://article/cf5ad199-d749-52e8-ba54-3e2009838201,https://www.nytimes.com/2025/07/23/technology/trump-ai-executive-orders.html,1269
4,Private equity firms like Blackstone are using their clients’ money to buy and build data centers to fuel the artifi...,By Maureen Farrell,article,Wall St. Is All In on A.I. Data Centers. But Are They the Next Bubble?,nyt://article/df74d392-81ab-5096-9b61-3fd65c50b72f,Business,1,B,2025-06-02T09:00:12Z,Business,Private equity firms like Blackstone are using their clients’ money to buy and build data centers to fuel the artifi...,The New York Times,,News,nyt://article/df74d392-81ab-5096-9b61-3fd65c50b72f,https://www.nytimes.com/2025/06/02/business/ai-data-centers-private-equity.html,1579
5,"Meet your artificial intelligence matchmakers. These A.I. tools are changing dating apps, so users don’t have to swi...",By Eli Tan,article,You Don’t Need to Swipe Right. A.I. Is Transforming Dating Apps.,nyt://article/9f75c386-cf24-51ef-8e2e-6adb1b16e2b5,Business,1,B,2025-11-03T15:20:51Z,Technology,"Meet your artificial intelligence matchmakers. These A.I. tools are changing dating apps, so users don’t have to swi...",The New York Times,,News,nyt://article/9f75c386-cf24-51ef-8e2e-6adb1b16e2b5,https://www.nytimes.com/2025/11/03/technology/ai-dating-apps.html,1202
6,President Trump’s plan to bar ships from entering or leaving Iranian ports has put markets on edge and added to glob...,"By Andrew Ross Sorkin, Bernhard Warner, Sarah Kessler, Michael J. de la Merced, Niko Gallogly and Brian O’Keefe",article,Investors Brace for a New Hormuz Blockade Threat,nyt://article/ea28affb-74fe-5915-a7c0